# CommunityForensics workflow

This standalone Kaggle notebook has separate stages. First, it downloads the configured CommunityForensics shards, extracts features, creates prompt-grouped train/calibration/test partitions, trains and evaluates the 11 generator labels plus `real_camera`, and selects 10 held-out images per class. Second, it packages the model, test images, and generated reports into separate ZIP files. Heavy dataset processing and training run in Kaggle; local computers only need to download and install those outputs.

Source code is organized by function in adjacent `.py` files and embedded here for Kaggle execution. The notebook does not import the local application package or install this repository. Commercial rows marked `test` are used in an internal grouped evaluation, not an official benchmark score. Run the training stage before packaging; the packaging stage can be rerun while the training artifacts remain in the Kaggle session.


## Run settings


In [ ]:
"""Shared constants for the standalone CommunityForensics Kaggle workflow."""
from __future__ import annotations

from pathlib import Path
import numpy as np

SHARDS = [
    (
        "OwensLab/CommunityForensics",
        "HFCF_Commercial_0.parquet",
        "https://huggingface.co/datasets/OwensLab/CommunityForensics/resolve/main/data/commercial/HFCF_Commercial_0.parquet",
    ),
    (
        "OwensLab/CommunityForensics",
        "HFCF_Commercial_1.parquet",
        "https://huggingface.co/datasets/OwensLab/CommunityForensics/resolve/main/data/commercial/HFCF_Commercial_1.parquet",
    ),
    (
        "OwensLab/CommunityForensics-Small",
        "HFCF_small_174.parquet",
        "https://huggingface.co/datasets/OwensLab/CommunityForensics-Small/resolve/main/data/HFCF_small_174.parquet",
    ),
]

TARGET_MODELS = {
    "dalle2": "dalle2",
    "dalle3": "dalle3",
    "midjourneyv52": "midjourney_v5_2",
    "midjourneyv61": "midjourney_v6_1",
    "fireflyimage2": "firefly_image2",
    "fireflyimage3": "firefly_image3",
    "fluxdev": "flux_dev",
    "fluxschnell": "flux_schnell",
    "ideogramv1": "ideogram_v1",
    "ideogramv2": "ideogram_v2",
    "imagen3": "imagen3",
}
REAL_MODELS = {"landscapeshq", "coco", "ffhq", "vision"}
TARGET_LABELS = sorted([*TARGET_MODELS.values(), "real_camera"])
MAX_PER_CLASS = 100
MAX_UNKNOWN_PER_MODEL = 25
MAX_UNKNOWN_TOTAL = 200
IMAGE_EXPORT_PER_CLASS = 10
CROP_SIZE = 512
PCA_COMPONENTS = 128
RANGE_CHUNK_BYTES = 64 * 1024 * 1024
SRM_KERNEL = np.array([[-1, 2, -1], [2, -4, 2], [-1, 2, -1]], dtype=np.float64)
LUMA = np.array([0.299, 0.587, 0.114], dtype=np.float64)

WORK = Path("/kaggle/working")
RAW_DIR = WORK / "communityforensics_raw"
OUTPUT_DIR = WORK / "genaifp-results"
IMAGE_DIR = OUTPUT_DIR / "extracted-images"
IMAGE_CANDIDATE_DIR = WORK / "communityforensics-image-candidates"
FEATURES_PATH = OUTPUT_DIR / "features_communityforensics_real.parquet"


## Dataset access and labels


In [ ]:
"""Dataset labels, image-payload helpers, and resumable Parquet shard downloads."""
from __future__ import annotations

import hashlib
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pyarrow.parquet as pq
import requests



def normalize_model(value: object) -> str:
    return re.sub(r"[^a-z0-9]", "", str(value or "").split("/")[-1].lower())

def image_bytes(value: object) -> bytes:
    if isinstance(value, bytes):
        return value
    if isinstance(value, (bytearray, memoryview)):
        return bytes(value)
    if isinstance(value, dict) and value.get("bytes"):
        return image_bytes(value["bytes"])
    raise ValueError(f"Unsupported or empty image_data payload: {type(value).__name__}")

def label_record(record: dict[str, object]) -> tuple[str, bool] | None:
    model = normalize_model(record.get("model_name"))
    source_label = str(record.get("label", ""))
    if model in TARGET_MODELS:
        return TARGET_MODELS[model], False
    if model in REAL_MODELS or source_label == "0":
        return "real_camera", False
    if source_label == "1" and model:
        return "unknown", True
    return None

def download_range(url: str, start: int, end: int, path: Path) -> None:
    expected = end - start + 1
    last_error: Exception | None = None
    for attempt in range(5):
        try:
            with requests.get(
                url,
                headers={"Range": f"bytes={start}-{end}"},
                stream=True,
                timeout=(30, 120),
            ) as response:
                response.raise_for_status()
                content_range = response.headers.get("Content-Range", "")
                if response.status_code != 206 or not content_range.startswith(f"bytes {start}-{end}/"):
                    raise OSError(f"Unexpected range response: {response.status_code} {content_range}")
                written = 0
                with path.open("r+b") as handle:
                    handle.seek(start)
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            handle.write(chunk)
                            written += len(chunk)
                if written != expected:
                    raise OSError(f"Range wrote {written} bytes; expected {expected}")
                return
        except (requests.RequestException, OSError) as exc:
            last_error = exc
            time.sleep(min(2**attempt, 8))
    raise OSError(f"Range {start}-{end} failed after retries: {last_error}")

def download_shard(url: str, output_path: Path) -> None:
    with requests.Session() as session:
        response = session.head(url, allow_redirects=True, timeout=30)
        response.raise_for_status()
        size = int(response.headers["Content-Length"])
        final_url = response.url
    if output_path.is_file() and output_path.stat().st_size == size:
        try:
            pq.ParquetFile(output_path).metadata.num_rows
            print(f"Reusing {output_path.name} ({size / 1e9:.2f} GB)")
            return
        except Exception:
            output_path.unlink(missing_ok=True)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("wb") as handle:
        handle.truncate(size)
    ranges = [
        (start, min(start + RANGE_CHUNK_BYTES, size) - 1)
        for start in range(0, size, RANGE_CHUNK_BYTES)
    ]
    print(f"Downloading {output_path.name} ({size / 1e9:.2f} GB) in {len(ranges)} ranges")
    with ThreadPoolExecutor(max_workers=12) as pool:
        futures = [pool.submit(download_range, final_url, start, end, output_path) for start, end in ranges]
        for future in as_completed(futures):
            future.result()
    pq.ParquetFile(output_path).metadata.num_rows

def group_id(prompt: object, encoded: bytes) -> str:
    normalized = " ".join(str(prompt or "").casefold().split())
    value = f"prompt:{normalized}" if normalized else "image:" + hashlib.sha256(encoded).hexdigest()
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


## Image and feature extraction


In [ ]:
"""Image preprocessing and visual/spatial-frequency feature extraction."""
from __future__ import annotations

import io

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageFilter
from scipy.ndimage import convolve
from scipy.optimize import minimize_scalar
from scipy.fft import fft2, fftshift



def center_crop(rgb: np.ndarray, size: int = CROP_SIZE) -> np.ndarray:
    if rgb.ndim == 2:
        rgb = np.repeat(rgb[:, :, None], 3, axis=2)
    if rgb.ndim == 3 and rgb.shape[2] == 1:
        rgb = np.repeat(rgb, 3, axis=2)
    rgb = rgb[:, :, :3]
    height, width = rgb.shape[:2]
    pad_y, pad_x = max(0, size - height), max(0, size - width)
    if pad_y or pad_x:
        top, left = pad_y // 2, pad_x // 2
        rgb = np.pad(
            rgb,
            ((top, pad_y - top), (left, pad_x - left), (0, 0)),
            mode="reflect" if min(height, width) > 1 else "edge",
        )
        height, width = rgb.shape[:2]
    top, left = (height - size) // 2, (width - size) // 2
    return rgb[top : top + size, left : left + size]

def build_spectral_grids() -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    y, x = np.ogrid[:CROP_SIZE, :CROP_SIZE]
    radius = np.floor(np.sqrt((y - CROP_SIZE // 2) ** 2 + (x - CROP_SIZE // 2) ** 2)).astype(np.int32)
    angle = np.degrees(np.arctan2(y - CROP_SIZE // 2, x - CROP_SIZE // 2)) + 180
    quadrant = np.clip((angle // 90).astype(np.int32), 0, 3)
    triangle = np.triu_indices(CROP_SIZE)
    return radius, quadrant, triangle

def spectral_features(rgb: np.ndarray, grids: tuple[np.ndarray, np.ndarray, np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    radius_grid, quadrant_grid, triangle = grids
    luminance = np.dot(rgb.astype(np.float64), LUMA)
    residual = convolve(luminance, SRM_KERNEL, mode="reflect").astype(np.float32)
    shifted = fftshift(fft2(residual))
    log_power = np.log1p(np.abs(shifted) ** 2).astype(np.float32)
    radius = radius_grid.ravel()
    quadrant = quadrant_grid.ravel()
    power = log_power.ravel()
    r_max = CROP_SIZE // 2 - 1
    valid = (radius >= 1) & (radius <= r_max)
    radial = radius[valid]
    sector = quadrant[valid]
    values = power[valid]
    counts = np.bincount(radial, minlength=r_max + 1)[1 : r_max + 1]
    sums = np.bincount(radial, weights=values, minlength=r_max + 1)[1 : r_max + 1]
    rapsd = np.divide(sums, counts, out=np.zeros_like(sums, dtype=np.float64), where=counts > 0)
    azimuthal = np.zeros((4, r_max), dtype=np.float64)
    for sector_id in range(4):
        selected = sector == sector_id
        bins = radial[selected]
        p = values[selected]
        count = np.bincount(bins, minlength=r_max + 1)[1 : r_max + 1]
        total = np.bincount(bins, weights=p, minlength=r_max + 1)[1 : r_max + 1]
        azimuthal[sector_id] = np.divide(total, count, out=np.zeros_like(total), where=count > 0)
    scalars = scalar_features(rapsd, azimuthal)
    flat = log_power[triangle].astype(np.float32)
    return flat, scalars

def scalar_features(rapsd: np.ndarray, azimuthal: np.ndarray) -> np.ndarray:
    spectrum = rapsd.astype(np.float64)
    total = spectrum.sum()
    if total <= 0:
        return np.zeros(12, dtype=np.float32)
    normalized = spectrum / total
    radii = np.arange(1, len(normalized) + 1, dtype=np.float64)
    peak_average = float(normalized.max() / max(normalized.mean(), 1e-18))
    peak_radius = int(np.argmax(normalized[: len(normalized) // 2]))
    fit_start = min(max(peak_radius, len(normalized) // 2), len(normalized) - 10)
    if fit_start < len(normalized) - 5:
        slope, intercept = np.polyfit(
            np.log(radii[fit_start:]), np.log(normalized[fit_start:] + 1e-18), 1
        )
        alpha, beta = float(-slope), float(intercept)
    else:
        alpha, beta = 0.0, 0.0
    edges = np.linspace(1, len(normalized) + 1, 9).astype(int)
    energies = [float(normalized[edges[i] - 1 : edges[i + 1] - 1].sum()) for i in range(8)]
    sector_sums = azimuthal.sum(axis=1, keepdims=True)
    sector_norm = np.divide(azimuthal, sector_sums, out=np.zeros_like(azimuthal), where=sector_sums > 0)
    directional_variance = float(np.var(sector_norm, axis=0).mean())
    return np.asarray([peak_average, alpha, beta, *energies, directional_variance], dtype=np.float32)

def create_visual_model() -> tuple[torch.nn.Module, torch.device]:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    try:
        model = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14", pretrained=True)
    except Exception as exc:
        raise RuntimeError(
            "Could not load pretrained DINOv2 through torch.hub. Enable Kaggle internet access."
        ) from exc
    model.to(device).eval()
    return model, device

def visual_features(rgb: np.ndarray, model: torch.nn.Module, device: torch.device) -> np.ndarray:
    image = Image.fromarray(rgb).resize((224, 224), Image.Resampling.BICUBIC)
    tensor = torch.from_numpy(np.asarray(image, dtype=np.float32).copy()).permute(2, 0, 1) / 255.0
    mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32)[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32)[:, None, None]
    tensor = ((tensor - mean) / std).unsqueeze(0).to(device)
    vector = model(tensor)
    vector = F.normalize(vector, p=2, dim=-1)
    return vector[0].detach().cpu().numpy().astype(np.float32)

def image_feature(encoded: bytes, model: torch.nn.Module, device: torch.device, grids) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    with Image.open(io.BytesIO(encoded)) as opened:
        opened.load()
        rgb = center_crop(np.asarray(opened.convert("RGB"), dtype=np.uint8))
    flat, scalars = spectral_features(rgb, grids)
    visual = visual_features(rgb, model, device)
    return flat, visual, scalars

def degraded_bytes(encoded: bytes, index: int) -> tuple[bytes, str]:
    with Image.open(io.BytesIO(encoded)) as opened:
        image = opened.convert("RGB")
    buffer = io.BytesIO()
    if index % 3 == 0:
        image.save(buffer, format="JPEG", quality=18, optimize=True)
        kind = "severe_jpeg"
    elif index % 3 == 1:
        width, height = image.size
        tiny = image.resize((max(8, width // 8), max(8, height // 8)), Image.Resampling.BILINEAR)
        tiny.resize((width, height), Image.Resampling.BICUBIC).save(buffer, format="JPEG", quality=55)
        kind = "severe_downscale"
    else:
        image.filter(ImageFilter.GaussianBlur(radius=8)).save(buffer, format="PNG")
        kind = "severe_blur"
    return buffer.getvalue(), kind


## Original-image sample export


In [ ]:
"""Export a deterministic, held-out image set with source provenance."""
from __future__ import annotations

import csv
import hashlib
import io
import math
import re
import shutil
from pathlib import Path

from PIL import Image, ImageDraw, ImageFont



def write_image_export(
    rows: list[dict[str, object]], test_indices, *, seed: int = 42
) -> tuple[Path, dict[str, int]]:
    """Write exactly N distinct original test images per class and a provenance manifest."""
    if IMAGE_DIR.exists():
        shutil.rmtree(IMAGE_DIR)
    IMAGE_DIR.mkdir(parents=True, exist_ok=True)

    candidates: dict[str, list[dict[str, object]]] = {label: [] for label in TARGET_LABELS}
    for index in test_indices:
        row = rows[int(index)]
        label = str(row.get("label", ""))
        path = str(row.get("image_path", ""))
        if bool(row.get("is_ood")) or label not in candidates or not path:
            continue
        if Path(path).is_file():
            candidates[label].append(row)

    manifest: list[dict[str, object]] = []
    counts: dict[str, int] = {}
    for label in TARGET_LABELS:
        ordered = sorted(
            candidates[label],
            key=lambda row: hashlib.sha256(
                f"{seed}:{row.get('group_id', '')}:{row.get('image_name', '')}".encode("utf-8")
            ).hexdigest(),
        )
        selected: list[dict[str, object]] = []
        seen_hashes: set[str] = set()
        for row in ordered:
            image_path = Path(str(row["image_path"]))
            encoded = image_path.read_bytes()
            digest = hashlib.sha256(encoded).hexdigest()
            if digest in seen_hashes:
                continue
            try:
                with Image.open(io.BytesIO(encoded)) as image:
                    image.verify()
            except Exception:
                continue
            seen_hashes.add(digest)
            selected.append({**row, "_encoded": encoded, "_sha256": digest})
            if len(selected) == IMAGE_EXPORT_PER_CLASS:
                break
        if len(selected) != IMAGE_EXPORT_PER_CLASS:
            raise ValueError(
                f"Test split has {len(selected)} usable distinct images for {label}; "
                f"need {IMAGE_EXPORT_PER_CLASS}. Increase the class sample cap or revise the split."
            )

        for position, row in enumerate(selected, start=1):
            encoded = bytes(row["_encoded"])
            try:
                with Image.open(io.BytesIO(encoded)) as image:
                    extension = (image.format or "PNG").lower()
            except Exception as exc:
                raise ValueError(f"Could not read selected test image for {label}: {exc}") from exc
            extension = {"jpeg": "jpg", "tiff": "tif"}.get(extension, extension)
            original_name = Path(str(row.get("image_name") or "image")).stem
            safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", original_name)[:80] or "image"
            filename = f"{position:02d}_{safe_name}.{extension}"
            destination = IMAGE_DIR / label / filename
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(encoded)
            manifest.append(
                {
                    "file": (Path("images") / label / filename).as_posix(),
                    "label": label,
                    "partition": "internal_test",
                    "dataset_id": row.get("dataset_id", ""),
                    "source_shard": row.get("source_shard", ""),
                    "source_split": row.get("source_split", ""),
                    "source_label": row.get("source_label", ""),
                    "model_name": row.get("model_name", ""),
                    "real_source": row.get("real_source", ""),
                    "subset": row.get("subset", ""),
                    "original_image_name": row.get("image_name", ""),
                    "group_id": row.get("group_id", ""),
                    "bytes": len(encoded),
                    "sha256": row["_sha256"],
                }
            )
        counts[label] = len(selected)

    manifest_path = OUTPUT_DIR / "communityforensics-image-manifest.csv"
    fields = list(manifest[0].keys())
    with manifest_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(manifest)
    sheet_path = OUTPUT_DIR / "communityforensics-image-contact-sheet.png"
    make_contact_sheet(manifest, IMAGE_DIR, sheet_path)
    return manifest_path, counts


def make_contact_sheet(manifest: list[dict[str, object]], image_root: Path, output: Path) -> None:
    columns, cell_w, image_h, label_h = 5, 240, 210, 42
    rows = math.ceil(len(manifest) / columns)
    canvas = Image.new("RGB", (columns * cell_w, rows * (image_h + label_h)), "#eeeeee")
    draw = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 14)
    except OSError:
        font = ImageFont.load_default()
    for index, item in enumerate(manifest):
        relative = Path(str(item["file"]).replace("images/", "", 1))
        with Image.open(image_root / relative) as original:
            image = original.convert("RGB")
            image.thumbnail((cell_w - 12, image_h - 8), Image.Resampling.LANCZOS)
        col, row = index % columns, index // columns
        x = col * cell_w + (cell_w - image.width) // 2
        y = row * (image_h + label_h) + (image_h - image.height) // 2
        canvas.paste(image, (x, y))
        draw.text(
            (col * cell_w + 5, row * (image_h + label_h) + image_h + 4),
            f"{item['label']} · {item['model_name']}",
            fill="#222222",
            font=font,
        )
    canvas.save(output, optimize=True)


## Dataset scan and feature table


In [ ]:
"""Scan source shards, apply sample caps, export examples, and write feature Parquet."""
from __future__ import annotations

import hashlib
import io
from collections import Counter

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from PIL import Image



def build_feature_rows(model: torch.nn.Module, device: torch.device) -> tuple[list[dict[str, object]], dict[str, object]]:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    grids = build_spectral_grids()
    rows: list[dict[str, object]] = []
    target_counts: Counter[str] = Counter()
    unknown_counts: Counter[str] = Counter()
    seen_images: set[str] = set()
    degraded_index = 0
    malformed = 0
    IMAGE_CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)

    for dataset_id, shard_name, url in SHARDS:
        path = RAW_DIR / shard_name
        download_shard(url, path)
        parquet = pq.ParquetFile(path)
        columns = [
            column
            for column in ("image_name", "image_data", "model_name", "prompt", "real_source", "subset", "split", "label")
            if column in parquet.schema_arrow.names
        ]
        if "image_data" not in columns or "model_name" not in columns:
            raise ValueError(f"{shard_name} does not have image_data/model_name columns")
        print(f"Scanning {shard_name}: {parquet.metadata.num_rows:,} rows")
        for batch in parquet.iter_batches(batch_size=8, columns=columns):
            for record in batch.to_pylist():
                classified = label_record(record)
                if classified is None:
                    continue
                label, is_ood = classified
                model_key = normalize_model(record.get("model_name"))
                if is_ood:
                    if unknown_counts[model_key] >= MAX_UNKNOWN_PER_MODEL or sum(unknown_counts.values()) >= MAX_UNKNOWN_TOTAL:
                        continue
                elif target_counts[label] >= MAX_PER_CLASS:
                    continue
                try:
                    encoded = image_bytes(record.get("image_data"))
                    digest = hashlib.sha256(encoded).hexdigest()
                    if digest in seen_images:
                        continue
                    with Image.open(io.BytesIO(encoded)) as check:
                        check.verify()
                    seen_images.add(digest)
                    prompt = str(record.get("prompt") or "")
                    image_path = None
                    if not is_ood:
                        image_path = IMAGE_CANDIDATE_DIR / label / f"{digest}.image"
                        image_path.parent.mkdir(parents=True, exist_ok=True)
                        image_path.write_bytes(encoded)
                    common = {
                        "label": label,
                        "is_ood": is_ood,
                        "sample_kind": "unsupported_generator" if is_ood else "clean",
                        "group_id": group_id(prompt, encoded),
                        "image_name": str(record.get("image_name") or ""),
                        "model_name": str(record.get("model_name") or ""),
                        "prompt": prompt,
                        "real_source": str(record.get("real_source") or ""),
                        "subset": str(record.get("subset") or ""),
                        "source_shard": shard_name,
                        "source_split": str(record.get("split") or ""),
                        "source_label": str(record.get("label") or ""),
                        "dataset_id": dataset_id,
                        "dataset_kind": "communityforensics",
                        "image_path": str(image_path) if image_path else "",
                    }
                    flat, visual, scalars = image_feature(encoded, model, device, grids)
                    rows.append({**common, "flat_spectrum": flat, "visual_features": visual, "scalar_12": scalars})
                    if is_ood:
                        unknown_counts[model_key] += 1
                    else:
                        target_counts[label] += 1
                        degraded, degradation_kind = degraded_bytes(encoded, degraded_index)
                        degraded_index += 1
                        d_flat, d_visual, d_scalars = image_feature(degraded, model, device, grids)
                        rows.append(
                            {
                                **common,
                                "is_ood": True,
                                "sample_kind": degradation_kind,
                                "flat_spectrum": d_flat,
                                "visual_features": d_visual,
                                "scalar_12": d_scalars,
                            }
                        )
                except Exception as exc:
                    malformed += 1
                    if malformed <= 20:
                        print(f"Skipped {shard_name}/{record.get('image_name', '?')}: {exc}")
        path.unlink(missing_ok=True)
        print(f"Finished {shard_name}; classes={dict(target_counts)}; unknown={sum(unknown_counts.values())}")

    missing = [label for label in TARGET_LABELS if target_counts[label] < 2]
    if missing or degraded_index < 2:
        raise ValueError(
            f"Not enough usable rows; missing classes={missing}; "
            f"degraded OOD examples={degraded_index}"
        )
    feature_summary = {
        "dataset_id": "OwensLab/CommunityForensics + OwensLab/CommunityForensics-Small",
        "target_counts": dict(target_counts),
        "unknown_counts": dict(unknown_counts),
        "unknown_total": int(sum(unknown_counts.values())),
        "degraded_examples": degraded_index,
        "malformed_rows": malformed,
        "source_shards": [name for _, name, _ in SHARDS],
        "sampling": {
            "max_per_target_class": MAX_PER_CLASS,
            "max_unsupported_per_generator": MAX_UNKNOWN_PER_MODEL,
            "max_unsupported_total": MAX_UNKNOWN_TOTAL,
        },
        "original_source_split_counts": {
            split: int(sum(row["source_split"] == split and not row["is_ood"] for row in rows))
            for split in sorted({str(row["source_split"]) for row in rows if row.get("source_split")})
        },
    }
    write_feature_file(rows)
    return rows, feature_summary

def write_feature_file(rows: list[dict[str, object]]) -> None:
    fields = [
        "label", "is_ood", "sample_kind", "group_id", "image_name", "model_name", "prompt",
        "real_source", "subset", "source_shard", "source_split", "source_label", "dataset_id",
        "dataset_kind", "visual_features", "flat_spectrum", "scalar_12", "image_data",
    ]
    table_rows = []
    for row in rows:
        output = {key: row.get(key) for key in fields}
        for feature in ("visual_features", "flat_spectrum", "scalar_12"):
            output[feature] = np.asarray(output[feature], dtype=np.float32).tolist()
        output["image_data"] = None
        table_rows.append(output)
    table = pa.Table.from_pylist(table_rows)
    pq.write_table(table, FEATURES_PATH, compression="zstd", row_group_size=16)
    print(f"Feature table: {FEATURES_PATH} ({table.num_rows:,} rows)")


## Training, calibration, and evaluation


In [ ]:
"""Prompt-grouped training, calibration, OOD scoring, and held-out evaluation."""
from __future__ import annotations

import csv
import json
from pathlib import Path

import numpy as np
import xgboost as xgb
import joblib
from scipy.optimize import minimize_scalar
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler



def select_splits(rows: list[dict[str, object]], seed: int = 42) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    labels = np.asarray([str(row["label"]) for row in rows])
    is_ood = np.asarray([bool(row["is_ood"]) for row in rows])
    groups = np.asarray([str(row["group_id"]) for row in rows])
    strata = np.where(is_ood, "ood", labels)
    counts = [len(np.unique(groups[strata == value])) for value in np.unique(strata)]
    folds = min(5, min(counts))
    if folds < 3:
        raise ValueError("Need at least three independent prompt groups per class/OOD stratum")
    outer = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)
    for development, test in outer.split(np.zeros(len(rows)), strata, groups):
        if not set(TARGET_LABELS).issubset(set(labels[test][~is_ood[test]])) or not is_ood[test].any():
            continue
        inner_strata, inner_groups = strata[development], groups[development]
        inner_counts = [len(np.unique(inner_groups[inner_strata == value])) for value in np.unique(inner_strata)]
        inner_folds = min(5, min(inner_counts))
        if inner_folds < 2:
            continue
        inner = StratifiedGroupKFold(n_splits=inner_folds, shuffle=True, random_state=seed + 1)
        for train_local, calibration_local in inner.split(np.zeros(len(development)), inner_strata, inner_groups):
            train, calibration = development[train_local], development[calibration_local]
            if (
                set(TARGET_LABELS).issubset(set(labels[train][~is_ood[train]]))
                and set(TARGET_LABELS).issubset(set(labels[calibration][~is_ood[calibration]]))
                and is_ood[calibration].any()
            ):
                partitions = (train, calibration, test)
                group_sets = [set(groups[indices]) for indices in partitions]
                if any(group_sets[i] & group_sets[j] for i in range(3) for j in range(i + 1, 3)):
                    raise RuntimeError("Prompt groups crossed train, calibration, and test partitions")
                known_partition_indices = [indices[~is_ood[indices]] for indices in partitions]
                minimums = (2, 2, IMAGE_EXPORT_PER_CLASS)
                for partition_name, indices, minimum in zip(
                    ("train", "calibration", "test"), known_partition_indices, minimums
                ):
                    counts = {label: int(np.sum(labels[indices] == label)) for label in TARGET_LABELS}
                    insufficient = {label: count for label, count in counts.items() if count < minimum}
                    if insufficient:
                        raise ValueError(
                            f"{partition_name} partition needs at least {minimum} clean samples per "
                            f"target class; insufficient counts: {insufficient}"
                        )
                return train, calibration, test
    raise ValueError("Could not create prompt-group-disjoint train/calibration/test splits")

def hybrid_features(rows: list[dict[str, object]], indices: np.ndarray, scaler, pca) -> np.ndarray:
    raw = np.stack([rows[int(i)]["flat_spectrum"] for i in indices]).astype(np.float32)
    pca_values = pca.transform(scaler.transform(raw)).astype(np.float32)
    visual = np.stack([rows[int(i)]["visual_features"] for i in indices]).astype(np.float32)
    scalars = np.stack([rows[int(i)]["scalar_12"] for i in indices]).astype(np.float32)
    return np.concatenate((visual, pca_values, scalars), axis=1)

def model_probabilities(model, X: np.ndarray, temperature: float) -> np.ndarray:
    margins = np.asarray(model.predict(X, output_margin=True))
    if margins.ndim == 1:
        margins = margins.reshape(1, -1)
    logits = margins / max(temperature, 1e-6)
    logits -= logits.max(axis=1, keepdims=True)
    values = np.exp(logits)
    return values / values.sum(axis=1, keepdims=True)

def ood_features(model, X: np.ndarray, temperature: float, centroids: dict[str, np.ndarray], probabilities=None) -> np.ndarray:
    probabilities = probabilities if probabilities is not None else model_probabilities(model, X, temperature)
    confidence = probabilities.max(axis=1)
    entropy = -(probabilities * np.log(probabilities + 1e-12)).sum(axis=1) / np.log(max(probabilities.shape[1], 2))
    margins = np.asarray(model.predict(X, output_margin=True))
    if margins.ndim == 1:
        margins = margins.reshape(1, -1)
    scaled = margins / max(temperature, 1e-6)
    max_logits = scaled.max(axis=1)
    energy = -(max_logits + np.log(np.exp(scaled - max_logits[:, None]).sum(axis=1)))
    similarities = []
    for vector in X[:, :384]:
        unit = vector / (np.linalg.norm(vector) + 1e-12)
        similarities.append(max(float(np.dot(unit, center)) for center in centroids.values()))
    return np.column_stack((confidence, entropy, similarities, energy))

def calibrate_and_evaluate(rows: list[dict[str, object]], output: Path) -> dict[str, object]:
    train_idx, calibration_idx, test_idx = select_splits(rows)
    y = np.asarray([str(row["label"]) for row in rows])
    ood = np.asarray([bool(row["is_ood"]) for row in rows])
    train_known = train_idx[~ood[train_idx] & np.isin(y[train_idx], TARGET_LABELS)]
    calibration_known = calibration_idx[~ood[calibration_idx] & np.isin(y[calibration_idx], TARGET_LABELS)]
    calibration_ood = calibration_idx[ood[calibration_idx]]
    test_known = test_idx[~ood[test_idx] & np.isin(y[test_idx], TARGET_LABELS)]
    test_ood = test_idx[ood[test_idx]]
    if len(calibration_ood) < 2 or len(test_ood) < 2:
        raise ValueError("Need at least two OOD examples in calibration and test groups")

    raw_train = np.stack([rows[int(i)]["flat_spectrum"] for i in train_known]).astype(np.float32)
    scaler = StandardScaler()
    scaled_train = scaler.fit_transform(raw_train)
    pca = PCA(n_components=PCA_COMPONENTS, random_state=42, svd_solver="randomized")
    pca.fit(scaled_train)
    X_train = hybrid_features(rows, train_known, scaler, pca)
    encoder = LabelEncoder()
    y_train = encoder.fit_transform(y[train_known])
    model_params = {
        "objective": "multi:softprob",
        "eval_metric": "mlogloss",
        "n_estimators": 300,
        "max_depth": 5,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 3,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "random_state": 42,
        "n_jobs": 4,
        "tree_method": "hist",
        "verbosity": 0,
        "num_class": len(np.unique(y_train)),
    }
    model = xgb.XGBClassifier(
        **model_params,
    )
    model.fit(X_train, y_train)
    centroids = {}
    for class_name in encoder.classes_:
        vectors = X_train[y[train_known] == class_name, :384]
        center = vectors.mean(axis=0)
        centroids[str(class_name)] = center / (np.linalg.norm(center) + 1e-12)

    X_cal = hybrid_features(rows, calibration_known, scaler, pca)
    y_cal = encoder.transform(y[calibration_known])
    margins = np.asarray(model.predict(X_cal, output_margin=True))
    if margins.ndim == 1:
        margins = margins.reshape(1, -1)

    def nll(temperature: float) -> float:
        logits = margins / temperature
        logits -= logits.max(axis=1, keepdims=True)
        log_probs = logits - np.log(np.exp(logits).sum(axis=1, keepdims=True))
        return float(-np.mean(log_probs[np.arange(len(y_cal)), y_cal]))

    temperature = float(minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x)
    X_cal_ood = hybrid_features(rows, calibration_ood, scaler, pca)
    prob_cal = model_probabilities(model, X_cal, temperature)
    prob_cal_ood = model_probabilities(model, X_cal_ood, temperature)
    features_in = ood_features(model, X_cal, temperature, centroids, prob_cal)
    features_out = ood_features(model, X_cal_ood, temperature, centroids, prob_cal_ood)
    detector_X = np.vstack((features_in, features_out))
    detector_y = np.r_[np.zeros(len(features_in)), np.ones(len(features_out))]
    detector = make_pipeline(
        StandardScaler(), LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000)
    )
    detector.fit(detector_X, detector_y)
    in_scores = detector.predict_proba(features_in)[:, 1]
    threshold = float(np.quantile(in_scores, 0.95, method="higher"))

    X_test = hybrid_features(rows, test_known, scaler, pca)
    y_test = y[test_known]
    test_probs = model_probabilities(model, X_test, temperature)
    predictions = encoder.inverse_transform(test_probs.argmax(axis=1))
    X_test_ood = hybrid_features(rows, test_ood, scaler, pca)
    test_ood_scores = detector.predict_proba(ood_features(model, X_test_ood, temperature, centroids))[:, 1]
    result = {
        "test": {
            "accuracy": float(accuracy_score(y_test, predictions)),
            "macro_f1": float(f1_score(y_test, predictions, average="macro")),
            "per_class": {},
            "known_samples": int(len(test_known)),
            "ood_samples": int(len(test_ood)),
            "ood_recall": float(np.mean(test_ood_scores > threshold)),
            "ood_threshold": threshold,
        },
        "calibration": {
            "temperature": temperature,
            "validation_false_reject_rate": float(np.mean(in_scores > threshold)),
            "validation_ood_recall": float(np.mean(detector.predict_proba(features_out)[:, 1] > threshold)),
            "validation_ood_auroc": float(roc_auc_score(detector_y, detector.predict_proba(detector_X)[:, 1])),
        },
        "split_counts": {
            "train_known": int(len(train_known)),
            "calibration_known": int(len(calibration_known)),
            "calibration_ood": int(len(calibration_ood)),
            "test_known": int(len(test_known)),
            "test_ood": int(len(test_ood)),
            "train_groups": len(set(str(rows[int(i)]["group_id"]) for i in train_known)),
            "calibration_groups": len(set(str(rows[int(i)]["group_id"]) for i in calibration_idx)),
            "test_groups": len(set(str(rows[int(i)]["group_id"]) for i in test_idx)),
        },
    }
    for class_name in TARGET_LABELS:
        mask = y_test == class_name
        result["test"]["per_class"][class_name] = {
            "samples": int(mask.sum()),
            "accuracy": float(np.mean(predictions[mask] == class_name)) if mask.any() else None,
            "f1": float(f1_score(y_test, predictions, labels=[class_name], average="macro", zero_division=0)),
        }

    image_manifest, image_counts = write_image_export(rows, test_idx)
    result["manual_test_images"] = {
        "manifest": image_manifest.name,
        "per_class": image_counts,
        "total": int(sum(image_counts.values())),
    }

    model_path = output / "attribution_hybrid.json"
    model.save_model(model_path)
    joblib.dump(
        {
            "label_encoder": encoder,
            "class_names": encoder.classes_.tolist(),
            "n_classes": len(encoder.classes_),
            "params": model_params,
            "visual_centroids": centroids,
            "temperature": temperature,
            "probabilities_calibrated": True,
            "ood_detector": detector,
            "ood_threshold": threshold,
        },
        output / "attribution_hybrid.pkl",
    )
    joblib.dump(
        {
            "pca": pca,
            "scaler": scaler,
            "n_pca_components": PCA_COMPONENTS,
            "n_subbands": 8,
            "r_max": 255,
        },
        output / "feature_extractor.pkl",
    )
    importance_path = output / "feature_importance.csv"
    with importance_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["feature_index", "importance"])
        writer.writerows(enumerate(np.asarray(model.feature_importances_, dtype=float)))
    (output / "evaluation_summary.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result


## Training stage


In [ ]:
"""Orchestrate the standalone CommunityForensics Kaggle run."""
from __future__ import annotations

import json
import hashlib
import platform
import warnings
from datetime import UTC, datetime
import shutil
from pathlib import Path

import numpy as np
import torch



def main() -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle GPU accelerator before running this script")
    torch.set_num_threads(4)
    torch.manual_seed(42)
    np.random.seed(42)
    warnings.filterwarnings("ignore", category=UserWarning)
    for artifact in (
        WORK / "genaifp-model-bundle.zip",
        WORK / "genaifp-test-images.zip",
        WORK / "genaifp-reports.zip",
    ):
        artifact.unlink(missing_ok=True)
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    shutil.rmtree(RAW_DIR, ignore_errors=True)
    shutil.rmtree(IMAGE_CANDIDATE_DIR, ignore_errors=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    print("Python:", platform.python_version())
    print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    print("Loading DINOv2 pretrained visual model")
    vision_model, device = create_visual_model()
    rows, summary = build_feature_rows(vision_model, device)
    del vision_model
    torch.cuda.empty_cache()
    (OUTPUT_DIR / "import_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("Imported rows:", len(rows), "summary:", json.dumps(summary))
    evaluation = calibrate_and_evaluate(rows, OUTPUT_DIR)
    if evaluation["manual_test_images"]["total"] != len(TARGET_LABELS) * 10:
        raise RuntimeError("Expected exactly 10 held-out test images for every target class")
    dataset_digest = hashlib.sha256()
    with FEATURES_PATH.open("rb") as feature_file:
        for chunk in iter(lambda: feature_file.read(1024 * 1024), b""):
            dataset_digest.update(chunk)
    environment = {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "created_at": datetime.now(UTC).isoformat(),
        "entrypoint": "main.py (standalone; no local project package installed)",
    }
    (OUTPUT_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
    model_manifest = {
        "format_version": 1,
        "model_type": "hybrid_524d",
        "dataset_id": summary["dataset_id"],
        "dataset_kind": "communityforensics",
        "dataset_sha256": dataset_digest.hexdigest(),
        "evaluation_status": "research_holdout_complete",
        "evaluation_type": "internal_prompt_grouped_holdout",
        "evaluation_note": (
            "Commercial-source rows are marked test in the source dataset. These partitions are an "
            "internal grouped evaluation and are not the dataset's official benchmark score."
        ),
        "evaluated_on_heldout_groups": True,
        "grouping_method": "normalized_prompt_hash; exact-image-hash fallback when prompt is absent",
        "grouping_limitations": [
            "The dataset does not expose generation-batch IDs or unique source-photo IDs; prompt-disjoint groups cannot guarantee source-image-disjoint partitions."
        ],
        "classes": TARGET_LABELS,
        "feature_extractor": "feature_extractor.pkl",
        "feature_dimensions": {"visual": 384, "spectral_pca": 128, "scalar": 12},
        "visual_model": "facebookresearch/dinov2_vits14",
        "preprocessing": {"crop_size": CROP_SIZE, "color": "RGB", "crop_policy": "center-reflect-pad"},
        "source_shards": summary["source_shards"],
        "sample_counts": evaluation["split_counts"],
        "probabilities_calibrated": True,
        "temperature": evaluation["calibration"]["temperature"],
        "ood_threshold": evaluation["test"]["ood_threshold"],
        "ood_calibration": {
            "validation_false_reject_rate": evaluation["calibration"]["validation_false_reject_rate"],
            "validation_ood_recall": evaluation["calibration"]["validation_ood_recall"],
            "validation_ood_auroc": evaluation["calibration"]["validation_ood_auroc"],
        },
        "created_at": environment["created_at"],
        "kaggle_kernel": {"handle": "nahilr/genaifp-communityforensics-train"},
        "sampling": {
            "max_per_target_class": summary["sampling"]["max_per_target_class"],
            "max_unsupported_per_generator": summary["sampling"]["max_unsupported_per_generator"],
            "max_unsupported_total": summary["sampling"]["max_unsupported_total"],
            "unsupported_generator_examples_found": summary["unknown_total"],
            "degraded_examples_included": True,
            "original_image_bytes_retained_in_features": False,
        },
        "source_provenance": {
            "source_shard": summary["source_shards"],
            "source_split": sorted({str(row["source_split"]) for row in rows if row.get("source_split")}),
            "source_label": sorted({str(row["source_label"]) for row in rows if row.get("source_label")}),
            "subset": sorted({str(row["subset"]) for row in rows if row.get("subset")}),
            "model_name": sorted({str(row["model_name"]) for row in rows if row.get("model_name")}),
            "real_source": sorted({str(row["real_source"]) for row in rows if row.get("real_source")}),
            "original_source_split_counts": summary["original_source_split_counts"],
        },
        "original_image_bytes_retained_in_features": False,
    }
    manifest_path = OUTPUT_DIR / "attribution_hybrid.manifest.json"
    manifest_path.write_text(json.dumps(model_manifest, indent=2), encoding="utf-8")
    print("Evaluation:", json.dumps(evaluation, indent=2))
    shutil.rmtree(RAW_DIR, ignore_errors=True)
    shutil.rmtree(IMAGE_CANDIDATE_DIR, ignore_errors=True)
    FEATURES_PATH.unlink(missing_ok=True)
    print(f"Training artifacts are ready in: {OUTPUT_DIR}")
    print(f"Held-out sample images are ready in: {OUTPUT_DIR / 'extracted-images'}")


## Packaging stage


In [ ]:
"""Package a completed Kaggle training run into three portable ZIP files."""
from __future__ import annotations

import csv
import io
import shutil
import zipfile
from pathlib import Path


MODEL_FILES = {
    "attribution_hybrid.json",
    "attribution_hybrid.pkl",
    "attribution_hybrid.manifest.json",
    "feature_extractor.pkl",
}
REPORT_FILES = {
    "attribution_hybrid.manifest.json",
    "environment.json",
    "evaluation_summary.json",
    "feature_importance.csv",
    "import_summary.json",
}
MODEL_ARCHIVE = WORK / "genaifp-model-bundle.zip"
IMAGE_ARCHIVE = WORK / "genaifp-test-images.zip"
REPORT_ARCHIVE = WORK / "genaifp-reports.zip"


def _write_zip(destination: Path, files: list[tuple[Path, str]]) -> None:
    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.unlink(missing_ok=True)
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for source, archive_name in files:
            if not source.is_file():
                raise FileNotFoundError(f"Required run artifact is missing: {source}")
            archive.write(source, archive_name)
    with zipfile.ZipFile(temporary) as archive:
        if archive.testzip() is not None:
            raise RuntimeError(f"Archive validation failed: {destination.name}")
    temporary.replace(destination)


def _validate_existing_archives() -> bool:
    archives = (MODEL_ARCHIVE, IMAGE_ARCHIVE, REPORT_ARCHIVE)
    if not all(path.is_file() for path in archives):
        return False
    for path in archives:
        with zipfile.ZipFile(path) as archive:
            if archive.testzip() is not None:
                raise RuntimeError(f"Existing archive is corrupt: {path}")
    return True


def package_outputs() -> None:
    """Package a completed training stage; safe to repeat after a successful package."""
    if not OUTPUT_DIR.is_dir():
        if _validate_existing_archives():
            print("The three Kaggle output archives are already packaged and valid.")
            return
        raise FileNotFoundError(
            f"No training artifacts found at {OUTPUT_DIR}; run the training stage first."
        )

    model_paths = [(OUTPUT_DIR / name, name) for name in sorted(MODEL_FILES)]
    report_paths = [(OUTPUT_DIR / name, name) for name in sorted(REPORT_FILES)]
    _write_zip(MODEL_ARCHIVE, model_paths)
    _write_zip(REPORT_ARCHIVE, report_paths)

    image_manifest = OUTPUT_DIR / "communityforensics-image-manifest.csv"
    contact_sheet = OUTPUT_DIR / "communityforensics-image-contact-sheet.png"
    image_files = [
        (path, (Path("images") / path.relative_to(IMAGE_DIR)).as_posix())
        for path in sorted(IMAGE_DIR.rglob("*"))
        if path.is_file()
    ]
    image_files.extend(
        [
            (image_manifest, "manifest.csv"),
            (contact_sheet, "contact_sheet.png"),
        ]
    )
    _write_zip(IMAGE_ARCHIVE, image_files)
    with zipfile.ZipFile(IMAGE_ARCHIVE) as archive:
        image_names = [
            name for name in archive.namelist()
            if name.startswith("images/") and not name.endswith("/")
        ]
        rows = list(csv.DictReader(io.TextIOWrapper(archive.open("manifest.csv"), encoding="utf-8")))
        counts = {label: 0 for label in TARGET_LABELS}
        for row in rows:
            if row.get("label") in counts and row.get("partition") == "internal_test":
                counts[row["label"]] += 1
        expected = len(TARGET_LABELS) * IMAGE_EXPORT_PER_CLASS
        if len(image_names) != expected or len(rows) != expected or any(
            count != IMAGE_EXPORT_PER_CLASS for count in counts.values()
        ):
            raise RuntimeError(
                f"Test image archive must contain {IMAGE_EXPORT_PER_CLASS} held-out images per class; "
                f"found {len(image_names)} files and counts {counts}"
            )

    shutil.rmtree(RAW_DIR, ignore_errors=True)
    shutil.rmtree(IMAGE_CANDIDATE_DIR, ignore_errors=True)
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    for path in (MODEL_ARCHIVE, IMAGE_ARCHIVE, REPORT_ARCHIVE):
        print(f"Packaged: {path} ({path.stat().st_size / 1e6:.1f} MB)")


def run_package_outputs() -> None:
    package_outputs()


## Run the training stage

Run this cell to download/process the data, train/evaluate the model, and leave lightweight artifacts ready for packaging.


In [ ]:
main()


## Package the outputs

Run this after training. It creates separate model, test-image, and report archives. It can also be run on its own after a completed training stage in this Kaggle session.


In [ ]:
package_outputs()
